# RetailPulse: Exploratory Data Analysis & Modeling Sandbox
This notebook serves as the early development and exploratory workspace for the **RetailPulse** platform. 
We will explore the transactional, product, and inventory datasets, test feature engineering strategies, and build prototypes for our customer segmentation, churn prediction, and demand forecasting models.

In [ ]:
import os
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print("Libraries loaded successfully.")

## 1. Data Ingestion & Initial Inspection
Let's load the raw datasets generated from our ETL process and check their structures.

In [ ]:
data_dir = "../data"
sales = pd.read_csv(os.path.join(data_dir, "sales_raw.csv"))
products = pd.read_csv(os.path.join(data_dir, "products_raw.csv"))
customers = pd.read_csv(os.path.join(data_dir, "customers_raw.csv"))
inventory = pd.read_csv(os.path.join(data_dir, "inventory_raw.csv"))

print(f"Sales: {sales.shape} rows")
print(f"Products: {products.shape} rows")
print(f"Customers: {customers.shape} rows")
print(f"Inventory: {inventory.shape} rows")

In [ ]:
sales.head()

## 2. Exploratory Data Analysis (EDA)
Let's analyze general trends: revenue by category, transaction volume, and price distributions.

In [ ]:
# Join sales with product details
sales_m = sales.merge(products, on="ProductID", how="left")

# Revenue by category
rev_cat = sales_m.groupby("Category")["Sales"].sum().reset_index()
sns.barplot(data=rev_cat, x="Category", y="Sales", palette="muted")
plt.title("Total Revenue by Product Category")
plt.ylabel("Revenue ($)")
plt.show()

## 3. Customer RFM & Segmentation
We calculate Recency, Frequency, and Monetary (RFM) values for our customer base and build clustering models.

In [ ]:
sales_m["OrderDate"] = pd.to_datetime(sales_m["OrderDate"])
snapshot = sales_m["OrderDate"].max() + pd.Timedelta(days=1)

rfm = sales_m.groupby("CustomerID").agg({
    "OrderDate": lambda x: (snapshot - x.max()).days,
    "OrderID": "nunique",
    "Sales": "sum"
}).rename(columns={
    "OrderDate": "Recency",
    "OrderID": "Frequency",
    "Sales": "Monetary"
}).reset_index()

rfm.describe()

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

scaler = StandardScaler()
scaled_rfm = scaler.fit_transform(rfm[["Recency", "Frequency", "Monetary"]])

# Determine optimal K using elbow plot
inertia = []
for k in range(1, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(scaled_rfm)
    inertia.append(km.inertia_)

plt.plot(range(1, 8), inertia, marker="o")
plt.title("Elbow Method for Optimal Clusters (K)")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.show()

## 4. Churn Prediction Feature Engineering & Modeling
We label customers as Churned (inactive > 90 days) and engineer leak-free features based on purchase intervals and order details to train a Random Forest model.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_curve, auc

# Mark churn based on 90d inactivity
rfm["Churn"] = (rfm["Recency"] > 90).astype(int)

# Build leak-free behavioral features
# Note: Recency is excluded to prevent data leakage!
features = rfm[["Frequency", "Monetary"]].copy()
features["AOV"] = features["Monetary"] / features["Frequency"]

X_train, X_test, y_train, y_test = train_test_split(features, rfm["Churn"], test_size=0.2, random_state=42)
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

## 5. Time-Series Sales Forecasting
Aggregate daily sales and inspect the trend and seasonal patterns.

In [ ]:
daily_sales = sales_m.groupby("OrderDate")["Sales"].sum().reset_index().sort_values("OrderDate")
plt.plot(daily_sales["OrderDate"], daily_sales["Sales"], label="Daily Sales")
plt.title("Historical Daily Sales Trend")
plt.xlabel("Date")
plt.ylabel("Sales ($)")
plt.legend()
plt.show()